In [ ]:
# @title 📦 Setup — Run this cell first! { display-mode: "form" }
# --- Google Colab setup ---
import os
if os.path.exists('/content') and not os.path.exists('/content/stable_diffusion_carlos'):
    print('📥 Cloning repository...')
    os.system('git clone https://github.com/eth-bmai-fs26/stable_diffusion_carlos.git /content/stable_diffusion_carlos')
if os.path.exists('/content/stable_diffusion_carlos'):
    os.chdir('/content/stable_diffusion_carlos')
    print(f'📂 Working directory: {os.getcwd()}')

%matplotlib inline
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import math

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.facecolor'] = 'white'

# Okabe-Ito accessible color palette
COLORS = {
    'text': '#E69F00',
    'image': '#0072B2',
    'connection': '#56B4E9',
    'failure': '#D55E00',
    'crossmodal': '#CC79A7',
    'neutral': '#999999',
}

print("✅ Setup complete.")


In [ ]:
# @title 🔧 Helper Functions — Data & Visualization { display-mode: "form" }

SHAPES = ['circle', 'square', 'triangle']
COLORS_MAP = {
    'red': (230, 25, 25),
    'blue': (25, 25, 230),
    'green': (25, 204, 25),
}
SIZES = {'small': 5, 'large': 10}
POSITIONS = {
    'center': (16, 16),
    'offset': [(8, 8), (8, 24), (24, 8), (24, 24)],
}

def generate_shape(shape, color, size='large', position='center', img_size=32):
    """Generate a 32x32x3 image with a colored shape. Returns numpy array [0,1]."""
    img = Image.new('RGB', (img_size, img_size), (0, 0, 0))
    draw = ImageDraw.Draw(img)
    rgb = COLORS_MAP[color]
    radius = SIZES[size]
    if position == 'center':
        cx, cy = 16, 16
    else:
        rng = np.random.RandomState()
        cx, cy = POSITIONS['offset'][rng.randint(len(POSITIONS['offset']))]
    if shape == 'circle':
        draw.ellipse([cx-radius, cy-radius, cx+radius, cy+radius], fill=rgb)
    elif shape == 'square':
        draw.rectangle([cx-radius, cy-radius, cx+radius, cy+radius], fill=rgb)
    elif shape == 'triangle':
        pts = [(cx, cy-radius), (cx-radius, cy+radius), (cx+radius, cy+radius)]
        draw.polygon(pts, fill=rgb)
    return np.array(img, dtype=np.float32) / 255.0

def generate_dataset(include_size=False, include_position=False):
    """Generate shape dataset. Base: 9 classes (3 shapes x 3 colors)."""
    sizes = list(SIZES.keys()) if include_size else ['large']
    positions = ['center', 'offset'] if include_position else ['center']
    dataset = []
    for color in COLORS_MAP:
        for shape in SHAPES:
            for sz in sizes:
                for pos in positions:
                    img = generate_shape(shape, color, size=sz, position=pos)
                    label = f"{sz} {color} {shape}" if include_size else f"{color} {shape}"
                    dataset.append((img, label))
    return dataset

class ShapeDataset(torch.utils.data.Dataset):
    def __init__(self, include_size=False, include_position=False):
        raw = generate_dataset(include_size=include_size, include_position=include_position)
        self.images, self.labels, self.label_texts = [], [], []
        unique_labels = []
        for img, label in raw:
            if label not in unique_labels:
                unique_labels.append(label)
        self.label_to_idx = {l: i for i, l in enumerate(unique_labels)}
        for img, label in raw:
            self.images.append(torch.from_numpy(img).permute(2, 0, 1))
            self.labels.append(self.label_to_idx[label])
            self.label_texts.append(label)
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx], self.label_texts[idx]

def plot_image_grid(images, titles=None, ncols=3, figsize=None, suptitle=None):
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    if figsize is None: figsize = (ncols * 2.5, nrows * 2.5)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).flatten() if isinstance(axes, np.ndarray) else [axes]
    for i, ax in enumerate(axes):
        if i < n:
            img = images[i]
            if isinstance(img, torch.Tensor):
                img = img.permute(1, 2, 0).detach().cpu().numpy() if img.dim() == 3 and img.shape[0] in (1,3) else img.detach().cpu().numpy()
            ax.imshow(np.clip(img, 0, 1))
            if titles and i < len(titles): ax.set_title(titles[i], fontsize=12)
        ax.axis('off')
    if suptitle: fig.suptitle(suptitle, fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

def plot_similarity_matrix(sim_matrix, row_labels, col_labels, title='Cosine Similarity'):
    if isinstance(sim_matrix, torch.Tensor): sim_matrix = sim_matrix.detach().cpu().numpy()
    n_rows, n_cols = sim_matrix.shape
    fig, ax = plt.subplots(figsize=(max(6, n_cols*0.8), max(5, n_rows*0.8)))
    im = ax.imshow(sim_matrix, cmap='Blues', vmin=-1, vmax=1, aspect='auto')
    for i in range(n_rows):
        for j in range(n_cols):
            val = sim_matrix[i, j]
            color = 'white' if abs(val) > 0.5 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=10, color=color)
    ax.set_xticks(range(n_cols)); ax.set_xticklabels(col_labels, rotation=45, ha='right', fontsize=10)
    ax.set_yticks(range(n_rows)); ax.set_yticklabels(row_labels, fontsize=10)
    ax.set_title(title, fontsize=14, fontweight='bold')
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    return fig

def plot_training_loss(losses, title='Training Loss'):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(losses, color=COLORS['image'], linewidth=2)
    ax.set_xlabel('Epoch', fontsize=12); ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3); fig.tight_layout()
    return fig

def plot_embedding_space_2d(embeddings, labels, title='Embedding Space',
                            text_color=None, image_color=None,
                            connections=None):
    """Plot 2D PCA of embeddings. Optionally draw connections between pairs."""
    if isinstance(embeddings, torch.Tensor): embeddings = embeddings.detach().cpu().numpy()
    centered = embeddings - embeddings.mean(axis=0)
    U, S, Vt = np.linalg.svd(centered, full_matrices=False)
    proj = centered @ Vt[:2].T
    return proj

print("✅ Helpers loaded.")


In [ ]:
# @title 🏗️ Models — TextEncoder, ImageEncoder, MiniCLIP { display-mode: "form" }

VOCAB = ['red','blue','green','circle','square','triangle',
         'small','large','center','offset','<pad>','<unk>']
WORD_TO_IDX = {w: i for i, w in enumerate(VOCAB)}

def tokenize(text, max_len=4):
    tokens = [WORD_TO_IDX.get(w, WORD_TO_IDX['<unk>']) for w in text.lower().split()]
    if len(tokens) < max_len: tokens += [WORD_TO_IDX['<pad>']] * (max_len - len(tokens))
    return torch.tensor(tokens[:max_len], dtype=torch.long)

class TextEncoder(nn.Module):
    """Embedding -> mean pool -> MLP -> L2 normalize."""
    def __init__(self, vocab_size=12, embed_dim=32, hidden_dim=64, out_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=WORD_TO_IDX['<pad>'])
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)
    def forward(self, token_ids):
        x = self.embedding(token_ids)
        mask = (token_ids != WORD_TO_IDX['<pad>']).unsqueeze(-1).float()
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.normalize(x, dim=-1)

    def encode_tokens(self, token_ids):
        """Per-token embeddings for cross-attention (before pooling)."""
        x = self.embedding(token_ids)
        mask = (token_ids != WORD_TO_IDX['<pad>']).unsqueeze(-1).float()
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x * mask

class ImageEncoder(nn.Module):
    """Flatten -> MLP -> L2 normalize."""
    def __init__(self, input_dim=3072, out_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, out_dim)
    def forward(self, images):
        x = images.flatten(1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.normalize(x, dim=-1)

    def encode_tokens(self, token_ids):
        """Per-token embeddings for cross-attention (before pooling)."""
        x = self.embedding(token_ids)
        mask = (token_ids != WORD_TO_IDX['<pad>']).unsqueeze(-1).float()
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x * mask

class MiniCLIP(nn.Module):
    """MiniCLIP: jointly trained text + image encoders with InfoNCE loss."""
    def __init__(self):
        super().__init__()
        self.text_encoder = TextEncoder()
        self.image_encoder = ImageEncoder()
    def forward(self, images, token_ids):
        return self.text_encoder(token_ids), self.image_encoder(images)
    def compute_loss(self, text_emb, image_emb, tau=0.07):
        logits = torch.matmul(text_emb, image_emb.t()) / tau
        labels = torch.arange(len(logits), device=logits.device)
        return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels)) / 2
    def tokenize(self, text):
        return tokenize(text)

print("✅ Models loaded.")


In [ ]:
# @title 🔄 Training — CLIP training loop { display-mode: "form" }

def train_clip(model, dataset, epochs=100, lr=1e-3, tau=0.07,
               save_every=None):
    """Train MiniCLIP. Returns loss history and optionally saved states."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    saved_states = {}

    images = torch.stack([dataset[i][0] for i in range(len(dataset))])
    texts = [dataset[i][2] for i in range(len(dataset))]
    token_ids = torch.stack([tokenize(t) for t in texts])

    for epoch in range(epochs):
        optimizer.zero_grad()
        text_emb, image_emb = model(images, token_ids)
        loss = model.compute_loss(text_emb, image_emb, tau=tau)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

        if save_every and (epoch % save_every == 0 or epoch == epochs - 1):
            import copy
            saved_states[epoch] = copy.deepcopy(model.state_dict())

        if (epoch + 1) % 20 == 0:
            print(f'  Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')

    return losses, saved_states

print("✅ Training functions loaded.")


In [ ]:
# Runtime check — make sure setup cells have been run
try:
    _ = len(COLORS)
    _ = MiniCLIP
    print("✅ Ready to go!")
except NameError:
    print("⚠️ Please run the setup cells above (grey cells 1-4).")
    print("   Click the ▶ button on each grey cell at the top.")


---

# 🎬 Notebook 2: The Translator — CLIP

## Teaching Text and Images a Common Language


In [ ]:
# DEMO: A trained CLIP model can match text to images perfectly
# Watch the diagonal light up — each text matches its correct image

dataset = ShapeDataset()
clip_model = MiniCLIP()

# Load pre-trained weights if available, otherwise train
try:
    weights = torch.load('weights/full_pipeline.pt', map_location='cpu', weights_only=False)
    clip_model.text_encoder.load_state_dict(weights['clip_text'])
    clip_model.image_encoder.load_state_dict(weights['clip_image'])
    print("✅ Pre-trained CLIP loaded.")
except:
    print("Training CLIP from scratch (this takes ~10 seconds)...")
    train_clip(clip_model, dataset, epochs=100)
    print("✅ Training complete.")

clip_model.eval()
with torch.no_grad():
    images = torch.stack([dataset[i][0] for i in range(len(dataset))])
    labels = [dataset[i][2] for i in range(len(dataset))]
    token_ids = torch.stack([tokenize(t) for t in labels])
    text_embs, image_embs = clip_model(images, token_ids)
    sim = torch.matmul(text_embs, image_embs.t())

fig = plot_similarity_matrix(sim, labels, labels,
                              title='CLIP Similarity: Text vs Images (Pre-trained)')
plt.show()
print("The bright diagonal = CLIP learned that matching texts and images")
print("are the same concept. This is the common language.")


> **Co-pilot metaphor:** *The co-pilot is learning to read the map.* CLIP teaches the co-pilot (text) and driver (image) to speak the same language, so they can work together.


In [ ]:
# Watch CLIP learn: text and image embeddings migrate together
# At epoch 0: scattered. By epoch 100: pairs overlap.
from ipywidgets import interact, IntSlider

# Train a fresh CLIP model, saving state every 10 epochs
print("Pre-computing embeddings at each training stage...")
fresh_clip = MiniCLIP()
_, saved_states = train_clip(fresh_clip, dataset, epochs=100, save_every=10)

# Pre-compute all projections
cached_projections = {}
for epoch, state in saved_states.items():
    fresh_clip.load_state_dict(state)
    fresh_clip.eval()
    with torch.no_grad():
        t_embs = fresh_clip.text_encoder(token_ids)
        i_embs = fresh_clip.image_encoder(images)
        all_e = torch.cat([t_embs, i_embs], dim=0).numpy()
        centered = all_e - all_e.mean(axis=0)
        U, S, Vt = np.linalg.svd(centered, full_matrices=False)
        proj = centered @ Vt[:2].T
        cached_projections[epoch] = proj
print("✅ Ready! Drag the slider below.")

@interact(epoch=IntSlider(min=0, max=100, step=10, value=0,
                           description='Epoch:'))
def show_epoch(epoch):
    proj = cached_projections[epoch]
    n = len(labels)
    fig, ax = plt.subplots(figsize=(8, 6))

    # Text points (orange), Image points (blue)
    for i in range(n):
        ax.scatter(proj[i, 0], proj[i, 1], c=COLORS['text'], s=80,
                   marker='o', zorder=5, edgecolors='black', linewidth=0.5)
        ax.scatter(proj[n+i, 0], proj[n+i, 1], c=COLORS['image'], s=80,
                   marker='s', zorder=5, edgecolors='black', linewidth=0.5)
        # Connect matched pairs
        ax.plot([proj[i,0], proj[n+i,0]], [proj[i,1], proj[n+i,1]],
                color=COLORS['connection'], linewidth=1, alpha=0.5)
        ax.annotate(labels[i], (proj[i, 0], proj[i, 1]),
                    fontsize=8, ha='center', va='bottom',
                    textcoords="offset points", xytext=(0, 6))

    import matplotlib.patches as mpatches
    ax.legend(handles=[
        mpatches.Patch(color=COLORS['text'], label='Text embeddings'),
        mpatches.Patch(color=COLORS['image'], label='Image embeddings'),
    ], fontsize=10)
    ax.set_title(f'Embedding Space at Epoch {epoch}', fontsize=14, fontweight='bold')
    ax.set_xlabel('PC 1', fontsize=12); ax.set_ylabel('PC 2', fontsize=12)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


**Watch the pairs find each other.** At epoch 0, text points (orange circles) and image points (blue squares) are scattered randomly. As training progresses, each text point migrates toward its matching image point.

By epoch 100, each text sits right on top of its matching image. **They've learned a common language.**


In [ ]:
# BUILD: The two encoders that learn the common language
# TextEncoder: words -> 32 numbers (a "text address")
# ImageEncoder: pixels -> 32 numbers (an "image address")

# The TextEncoder architecture:
#   1. Look up each word in a learned dictionary (embedding)
#   2. Average the word vectors together
#   3. Pass through a small neural network
#   4. Normalize to unit length (so cosine similarity works)

# The ImageEncoder architecture:
#   1. Flatten all pixels into one long list (3072 numbers)
#   2. Pass through a small neural network
#   3. Normalize to unit length

# Both produce 32-number "addresses" in the same space!
print(f"TextEncoder parameters: {sum(p.numel() for p in clip_model.text_encoder.parameters()):,}")
print(f"ImageEncoder parameters: {sum(p.numel() for p in clip_model.image_encoder.parameters()):,}")
print(f"Total CLIP parameters: {sum(p.numel() for p in clip_model.parameters()):,}")


In [ ]:
# BUILD: The InfoNCE loss — how CLIP learns
# The key idea: matching pairs should be similar, non-matching should differ

def info_nce_loss(text_emb, image_emb, temperature=0.07):
    """The loss function that teaches CLIP.
    
    For each text, its matching image should have the HIGHEST similarity.
    Temperature controls how "picky" the model is.
    """
    # Similarity matrix: every text vs every image
    sim = torch.matmul(text_emb, image_emb.t()) / temperature
    # Target: text[0] matches image[0], text[1] matches image[1], etc.
    targets = torch.arange(len(sim))
    # Cross-entropy pushes diagonal values UP, off-diagonal DOWN
    loss = (F.cross_entropy(sim, targets) + F.cross_entropy(sim.t(), targets)) / 2
    return loss

# Quick demo
print("Loss function loaded. This is what makes CLIP work:")
print("  • High similarity for matching pairs → pushed UP")
print("  • Low similarity for non-matching pairs → pushed DOWN")


In [ ]:
# BUILD: Train MiniCLIP on our 9 text-image pairs
# Watch the loss decrease from ~2.2 (random) to near 0 (perfect matching)

my_clip = MiniCLIP()
print("Training MiniCLIP on 9 text-image pairs for 100 epochs...")
losses, _ = train_clip(my_clip, dataset, epochs=100)

fig = plot_training_loss(losses, title='CLIP Training: Loss Over Time')
plt.show()

# Check retrieval accuracy
my_clip.eval()
with torch.no_grad():
    t_e, i_e = my_clip(images, token_ids)
    sims = torch.matmul(t_e, i_e.t())
    correct = (sims.argmax(dim=1) == torch.arange(len(labels))).sum().item()
print(f"\nRetrieval accuracy: {correct}/{len(labels)} ({100*correct/len(labels):.0f}%)")


In [ ]:
# The result: our trained CLIP's similarity matrix
my_clip.eval()
with torch.no_grad():
    t_e, i_e = my_clip(images, token_ids)
    sim_trained = torch.matmul(t_e, i_e.t())

fig = plot_similarity_matrix(sim_trained, labels, labels,
                              title='Your Trained CLIP: Text vs Image Similarity')
# Add sky blue border for success
for spine in fig.axes[0].spines.values():
    spine.set_edgecolor(COLORS['connection'])
    spine.set_linewidth(3)
plt.show()
print("The diagonal is bright! Your CLIP learned to match each text to its image.")


In [ ]:
# YOUR TURN: Pick a text prompt and see which images CLIP thinks match
from ipywidgets import interact, Dropdown

@interact(prompt=Dropdown(options=labels, value=labels[0],
                           description='Prompt:'))
def show_retrieval(prompt):
    my_clip.eval()
    with torch.no_grad():
        q_emb = my_clip.text_encoder(tokenize(prompt).unsqueeze(0))
        all_img_embs = my_clip.image_encoder(images)
        sims = torch.matmul(q_emb, all_img_embs.t()).squeeze()

    # Top-3 matches
    top3 = sims.argsort(descending=True)[:3]
    fig, axes = plt.subplots(1, 4, figsize=(12, 3))

    # Text embedding bar chart
    q_np = q_emb.squeeze().numpy()
    axes[0].barh(range(len(q_np)), q_np, color=COLORS['text'])
    axes[0].set_title(f'Text: "{prompt}"', fontsize=11)
    axes[0].set_ylabel('Dimension')

    # Top 3 images
    for rank, idx in enumerate(top3):
        img = images[idx].permute(1, 2, 0).numpy()
        axes[rank+1].imshow(np.clip(img, 0, 1))
        axes[rank+1].set_title(f'#{rank+1}: {labels[idx]}\nsim={sims[idx]:.3f}',
                                fontsize=10)
        axes[rank+1].axis('off')

    fig.suptitle(f'CLIP Retrieval for "{prompt}"', fontsize=14, fontweight='bold')
    fig.tight_layout()
    plt.show()


In [ ]:
# STUMBLE: What happens if we train on only 3 pairs instead of 9?
# Let's find out!

# Create a tiny subset: just 3 classes
subset_indices = [0, 4, 8]  # red circle, blue square, green triangle
class SubsetDataset:
    def __init__(self, full_dataset, indices):
        self.data = [(full_dataset[i][0], full_dataset[i][1], full_dataset[i][2])
                     for i in indices]
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

subset = SubsetDataset(dataset, subset_indices)
subset_labels = [labels[i] for i in subset_indices]

tiny_clip = MiniCLIP()
print("Training on ONLY 3 pairs...")
losses_tiny, _ = train_clip(tiny_clip, subset, epochs=100)

# Show similarity on ALL 9 classes (but model only saw 3!)
tiny_clip.eval()
with torch.no_grad():
    t_e, i_e = tiny_clip(images, token_ids)
    sim_tiny = torch.matmul(t_e, i_e.t())

fig = plot_similarity_matrix(sim_tiny, labels, labels,
                              title='CLIP Trained on Only 3 Pairs')
for spine in fig.axes[0].spines.values():
    spine.set_edgecolor(COLORS['failure'])
    spine.set_linewidth(3)
plt.show()
print("❌ Only 3 bright spots on the diagonal. The rest is noise.")
print("   It memorized 3 pairs but learned NOTHING about the other 6.")


### 💡 The Key Insight: Data Size Matters

**3 examples = memorization. 9 examples = concepts.** Our toy model needs all 9 pairs to learn that "red" is a concept shared across shapes.

Real CLIP was trained on **400 million text-image pairs** scraped from the internet. That's how it learned concepts like "sunset," "cat," "architecture" — not by memorizing individual images, but by seeing enough examples to extract patterns.

> **Co-pilot metaphor:** A co-pilot who has only driven 3 routes memorizes those routes. A co-pilot who has driven 400 million routes understands *how roads work*.


In [ ]:
# FIX: Our full 9-pair model learned transferable concepts!
# "red square" text has moderate similarity to "red circle" image — shared color

my_clip.eval()
with torch.no_grad():
    t_e, i_e = my_clip(images, token_ids)
    sim = torch.matmul(t_e, i_e.t())

# Highlight interesting partial matches
print("Partial matches show the model learned CONCEPTS, not just pairs:\n")
interesting = [
    ('red square', 'red circle', "shared color: red"),
    ('blue circle', 'green circle', "shared shape: circle"),
    ('red triangle', 'blue triangle', "shared shape: triangle"),
]
for text_label, img_label, reason in interesting:
    ti = labels.index(text_label)
    ii = labels.index(img_label)
    s = sim[ti, ii].item()
    match_s = sim[ti, ti].item()
    print(f'  "{text_label}" ↔ "{img_label}" image: sim={s:.3f} ({reason})')
    print(f'    vs perfect match: sim={match_s:.3f}\n')

print("It learned that \'redness\' is a transferable concept.")
print("This is why CLIP needs LOTS of data — more pairs = richer concepts.")


### 🌍 Real-World Story: The CLIP Bias Problem

When researchers analyzed CLIP (trained on 400M internet image-text pairs), they found something troubling:

- Searching for **"CEO"** returned mostly images of white men
- Searching for **"criminal"** returned disproportionately images of Black men

**Why?** CLIP faithfully encodes the correlations in its training data. If the internet associates "CEO" with a particular demographic, CLIP learns that association as a "concept."

**Formal diagnosis:** Contrastive learning like InfoNCE doesn't filter training data — it faithfully encodes whatever correlations exist. Biased data → biased embeddings.


<div style="background-color: #FFF3CD; border-left: 4px solid #E69F00; padding: 15px; margin: 10px 0; border-radius: 4px;">

**💼 Manager's Briefing: Fine-Tuning CLIP**

The number of training pairs is the single biggest factor in CLIP quality:
- **Too few pairs** → memorization (our 3-pair experiment)
- **Enough pairs** → concept learning (our 9-pair experiment)
- **Massive pairs** → rich, transferable knowledge (real CLIP, 400M pairs)

**Key questions for your vendor:**
1. *"How many image-text pairs was your model trained on?"*
2. *"Have you tested for demographic bias in the text encoder?"*
3. *"If we fine-tune on our data, how do you prevent catastrophic forgetting?"* (When a model forgets old knowledge while learning new things)

</div>


---

## 📋 Co-Pilot Reference Card

| Notebook | Component | What It Does | Co-Pilot Metaphor |
|----------|-----------|-------------|-------------------|
| NB1 | Raw Ingredients | Images = tensors of numbers; text = embedding vectors; cosine sim measures closeness | "Here are the parts of the car and the map" |
| **NB2** | **CLIP (Translator)** | **Learns shared embedding space; contrastive loss pulls matching pairs together** | **"The co-pilot learns to read the map"** |

*Two rows down, three to go. Next: the engine that generates images — diffusion.*

<div style="background-color: #FFF3CD; border-left: 4px solid #E69F00; padding: 15px; margin: 10px 0; border-radius: 4px;">

**🔍 Practical Tool: Bias Auditing**

You can audit a vendor's model using the same cosine similarity tool you just used:
1. Pick a concept (e.g., "doctor", "engineer", "nurse")
2. Compute similarities to images across demographic categories
3. If similarity scores vary significantly by demographic → the model has encoded bias

This is exactly what researchers did to uncover CLIP's biases. You now have the technical foundation to do the same.

</div>
